# RFM Customer Segmentation

This notebook performs a professional RFM analysis on cleaned sales data to identify customer segments and deliver actionable marketing and retention insights.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

## Load cleaned sales dataset

In [2]:
data_path = Path('..') / 'output' / 'cleaned_sales.csv'
df = pd.read_csv(data_path, parse_dates=['Order Date'])
df.head()

,Category,City,Country/Region,Customer ID,Customer Name,Order Date,Order ID,Postal Code,Product ID,Product Name,...,Days to Ship Actual,Days to Ship Scheduled,Discount,Profit,Quantity,Sales,Sales Forecast,Order Year,Order Month,Profit Margin
0,Furniture,Henderson,United States,CG-12520,Claire Gute,2019-11-08,CA-2019-152156,42420,FUR-BO-10001798,Bush Somerset Collection Bookcase,...,3,3,0.0,42,2,262,392,2019,11,0.160305
1,Furniture,Henderson,United States,CG-12520,Claire Gute,2019-11-08,CA-2019-152156,42420,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",...,3,3,0.0,220,3,732,1096,2019,11,0.300546
2,Office Supplies,Los Angeles,United States,DV-13045,Darrin Van Huff,2019-06-12,CA-2019-138688,90036,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,...,4,3,0.0,7,2,15,22,2019,6,0.466667
3,Furniture,Fort Lauderdale,United States,SO-20335,Sean O'Donnell,2018-10-11,US-2018-108966,33311,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangular Table,...,7,6,45.0,-383,5,958,1434,2018,10,-0.399791
4,Office Supplies,Fort Lauderdale,United States,SO-20335,Sean O'Donnell,2018-10-11,US-2018-108966,33311,OFF-ST-10000760,Eldon Fold 'N Roll Cart System,...,7,6,20.0,3,2,22,33,2018,10,0.136364


## Build RFM table

In [3]:
reference_date = df['Order Date'].max() + pd.Timedelta(days=1)
rfm = df.groupby(['Customer ID', 'Customer Name'], as_index=False).agg(
    Recency=('Order Date', lambda x: (reference_date - x.max()).days),
    Frequency=('Order ID', 'nunique'),
    Monetary=('Sales', 'sum')
)

rfm['R_Score'] = pd.qcut(rfm['Recency'].rank(method='first'), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

def label_segment(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    if row['R_Score'] >= 4 and row['F_Score'] >= 3:
        return 'Loyal Customers'
    if row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'High Value'
    if row['R_Score'] <= 2 and row['F_Score'] <= 2:
        return 'At Risk'
    return 'Opportunity'

rfm['Segment'] = rfm.apply(label_segment, axis=1)
rfm.head()

,Customer ID,Customer Name,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,AA-10315,Alex Avila,185,5,5565,2,2,5,225,At Risk
1,AA-10375,Allen Armold,20,9,1056,5,5,2,552,Loyal Customers
2,AA-10480,Andrew Allen,260,4,1792,1,1,3,113,At Risk
3,AA-10645,Anna Andreadi,56,6,5087,3,3,5,335,Opportunity
4,AB-10015,Aaron Bergman,417,3,887,1,1,1,111,At Risk


## RFM Segment Summary

In [4]:
segment_counts = rfm['Segment'].value_counts().reset_index()
segment_counts.columns = ['Segment', 'Customer Count']
segment_counts

,Segment,Customer Count
0,Opportunity,295
1,At Risk,171
2,Loyal Customers,123
3,Champions,106
4,High Value,98


## Visualize customer segments

In [5]:
fig = px.bar(segment_counts, x='Segment', y='Customer Count', color='Segment', title='Customer Segment Distribution', template='plotly_white')
fig.show()

## RFM distribution overview

In [6]:
fig = px.scatter(rfm, x='Recency', y='Monetary', size='Frequency', color='Segment', hover_data=['RFM_Score'], title='Recency vs Monetary by Segment', template='plotly_white')
fig.show()

## Export RFM scores

In [7]:
output_path = Path('..') / 'output' / 'rfm_scores.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
rfm.to_csv(output_path, index=False)
print(f'RFM scores saved to: {output_path}')

RFM scores saved to: ..\output\rfm_scores.csv
